# SCC0957 — Prática de Ciências de Dados II: Dados da Dengue PySuS
* Vinícius de Moraes - 13749910
* Rômulo Ferreira da Silva - 13734326
* João Pedro Barbosa Madeira - 13683038
* Pedro Silva dos Santos - 12688431
* Thiago Pasquotto Tavares - 15490194

## Carregando as bibliotecas

In [10]:
!uv pip install pysus==1.0.1 -q
!uv pip install nbformat
!uv pip install plotly
!uv pip install matplotlib

Using Python 3.11.15 environment at: /home/vini/.venv
Checked 1 package in 8ms
Using Python 3.11.15 environment at: /home/vini/.venv
Checked 1 package in 6ms
Using Python 3.11.15 environment at: /home/vini/.venv
Checked 1 package in 10ms


In [11]:
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import plotly.express as px
from pysus import SINAN
import polars as pl

## Carregando os dados

In [12]:
sinan = SINAN().load() # Loads the files from DATASUS
files = sinan.get_files(dis_code=["DENG"])
parquet = sinan.download(files)

21702967it [00:00, 1799095624.25it/s] 


In [13]:
df = pl.scan_parquet(
    [str(p) for p in parquet],
    extra_columns="ignore",
    missing_columns="insert"
)

In [14]:
df.collect_schema()

Schema([('ID_MUNICIP', String),
        ('ID_UNIDADE', String),
        ('DT_NOTIFIC', String),
        ('CS_RACA', String),
        ('CS_ESCOLAR', String),
        ('NU_ANO', String),
        ('SEM_NOT', String),
        ('SG_UF_NOT', String),
        ('ID_REGIONA', String),
        ('DT_SIN_PRI', String),
        ('SEM_PRI', String),
        ('NU_IDADE', String),
        ('CS_SEXO', String),
        ('ID_MN_RESI', String),
        ('ID_RG_RESI', String),
        ('SG_UF', String),
        ('ID_PAIS', String),
        ('ID_DG_NOT', String),
        ('ID_EV_NOT', String),
        ('ANT_DT_INV', String),
        ('OCUPACAO', String),
        ('DENGUE', String),
        ('ANO', String),
        ('VACINADO', String),
        ('DT_DOSE', String),
        ('FEBRE', String),
        ('DT_FEBRE', String),
        ('DURACAO', String),
        ('LACO', String),
        ('CEFALEIA', String),
        ('EXANTEMA', String),
        ('DOR', String),
        ('PROSTACAO', String),
        ('MIALGIA',

## Preparando os dados 

**Número de notificações/dia**

In [15]:
result = (
    df
    .group_by("DT_NOTIFIC")
    .agg(pl.len().alias("Notificações"))
    .rename({"DT_NOTIFIC": "Dia"})
    .sort("Dia")
)

df_notificacoes = result.collect().to_pandas()

df_notificacoes['Dia'] = pd.to_datetime(df_notificacoes['Dia'], format='%Y%m%d', errors='coerce')

df_notificacoes = df_notificacoes.dropna(subset=['Dia'])

df_notificacoes

,Dia,Notificações
0,2000-01-01,13
1,2000-01-02,97
2,2000-01-03,341
3,2000-01-04,386
4,2000-01-05,422
...,...,...
9705,2026-07-29,130
9706,2026-07-30,142
9707,2026-07-31,124
9708,2026-08-01,59


**Sazonalidade dos dados**

In [16]:
df_notificacoes["Ano"] = df_notificacoes["Dia"].dt.year
df_notificacoes["Mês"] = df_notificacoes["Dia"].dt.month

df_sazonalidade = (
    df_notificacoes
    .groupby(["Ano", "Mês"])["Notificações"]
    .mean()
    .reset_index()
)

df_sazonalidade

,Ano,Mês,Notificações
0,2000,1,458.290323
1,2000,2,853.103448
2,2000,3,1037.516129
3,2000,4,963.266667
4,2000,5,795.774194
...,...,...,...
315,2026,4,2933.766667
316,2026,5,2675.419355
317,2026,6,1978.900000
318,2026,7,1292.193548


**Classificação final da doença**

In [17]:
classi_fin = (
    df
    .select("DENGUE")
    .collect()
    .to_pandas()
)

classi_fin = (
    df
    .select(
        pl.col("DENGUE")
        .str.strip_chars()
        .replace({
            "9": "Ignorado",
            "8": "Inconclusivo",
            "1": "Positivo",
            "2": "Negativo",
            "": "Ignorado"
        })
        .fill_null("Ignorado")
        .alias("DENGUE")
    )
    .collect()
    .to_pandas()
)

classi_fin

,DENGUE
0,Negativo
1,Negativo
2,Negativo
3,Negativo
4,Negativo
...,...
28152067,Ignorado
28152068,Ignorado
28152069,Ignorado
28152070,Ignorado


**Notificações por raça/etnia**

In [18]:
cs_raca = (
    df
    .select(
        pl.col("CS_RACA")
        .str.strip_chars()
        .replace({
            '': 'Ignorado',
            '1': 'Branca',
            '2': 'Preta',
            '3': 'Amarela',
            '4': 'Parda',
            '5': 'Indígena',
            '9': 'Ignorado',
            '@': 'Ignorado'
        })
        .fill_null("Ignorado")
        .alias("CS_RACA")
    )
    .collect()
    .to_pandas()
)

**Evolução dos casos**

In [19]:
df_evolucao = (
    df.select(
        pl.col('CON_EVOLUC')
        .str.strip_chars()
        .replace({
            '0': 'não se aplica',
            '1': 'cura',
            '2': 'óbito pelo agravo',
            '3': 'óbito por outras causas',
            '4': 'óbito em investigação',
            '9': 'ignorado',
            '': 'ignorado',
            ']': 'ignorado'
        })
        .fill_null('ignorado')
        .alias('CON_EVOLUC')
    )
    .collect()
    .to_pandas()
)

df_evolucao

,CON_EVOLUC
0,cura
1,cura
2,cura
3,cura
4,cura
...,...
28152067,ignorado
28152068,ignorado
28152069,ignorado
28152070,ignorado


**Número de notificações por estado**

In [31]:
df_estado = (
    df
    .with_columns(
        pl.col("UF")
        .cast(pl.String)
        .str.strip_chars()
        .replace({
            "11": "RO",
            "12": "AC",
            "13": "AM",
            "14": "RR",
            "15": "PA",
            "16": "AP",
            "17": "TO",
            "21": "MA",
            "22": "PI",
            "23": "CE",
            "24": "RN",
            "25": "PB",
            "26": "PE",
            "27": "AL",
            "28": "SE",
            "29": "BA",
            "31": "MG",
            "32": "ES",
            "33": "RJ",
            "35": "SP",
            "41": "PR",
            "42": "SC",
            "43": "RS",
            "50": "MS",
            "51": "MT",
            "52": "GO",
            "53": "DF",
            "0": None,
            "": None,
            "`": None
        })
    )
    .drop_nulls("UF")
    .group_by("UF")
    .agg(pl.len().alias("Notificações"))
    .rename({"UF": "Estado"})
    .sort("Estado")
    .collect()
    .to_pandas()
)

df_estado

,Estado,Notificações
0,AC,4366
1,AL,8729
2,AM,5431
3,AP,2942
4,BA,42985
5,CE,34112
6,DF,28355
7,ES,27151
8,GO,90136
9,MA,27234


## Visualizações

In [21]:
media = df_notificacoes['Notificações'].mean()

fig = px.line(
    df_notificacoes,
    x='Dia',
    y='Notificações'
)

fig.add_hline(
    y=media,
    line_dash='dash',
    annotation_text=f"Média: {media:.0f}",
    annotation_position="top left"
)

fig.update_traces(line=dict(width=1))

fig.show()

In [22]:
fig = px.line(
    df_sazonalidade,
    x="Mês",
    y="Notificações",
    color="Ano",
    markers=True,
    labels={
        "Mês": "Mês",
        "Notificações": "Notificações",
        "Ano": "Ano"
    },
    title="Sazonalidade das notificações por ano"
)

fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(1, 13)),
        ticktext=[
            "Jan", "Fev", "Mar", "Abr",
            "Mai", "Jun", "Jul", "Ago",
            "Set", "Out", "Nov", "Dez"
        ]
    )
)

fig.show()

In [23]:
from plotly.subplots import make_subplots

# Criar ano e mês
df_notificacoes["Ano"] = df_notificacoes["Dia"].dt.year
df_notificacoes["Mês"] = df_notificacoes["Dia"].dt.month

# Soma das notificações de cada mês de cada ano
df_mensal = (
    df_notificacoes
    .groupby(["Ano", "Mês"])["Notificações"]
    .sum()
    .reset_index()
)

nomes_meses = [
    "Janeiro", "Fevereiro", "Março", "Abril",
    "Maio", "Junho", "Julho", "Agosto",
    "Setembro", "Outubro", "Novembro", "Dezembro"
]

# Criar 12 gráficos
fig = make_subplots(
    rows=4,
    cols=3,
    subplot_titles=nomes_meses
)

for mes in range(1, 13):

    dados = df_mensal[df_mensal["Mês"] == mes]

    # Média histórica daquele mês
    media = dados["Notificações"].mean()

    linha = go.Scatter(
        x=dados["Ano"],
        y=dados["Notificações"],
        mode="lines+markers",
        name="Notificações",
        showlegend=False
    )

    media_linha = go.Scatter(
        x=dados["Ano"],
        y=[media] * len(dados),
        mode="lines",
        name="Média",
        line=dict(dash="dash"),
        showlegend=False
    )

    linha_idx = (mes - 1) // 3 + 1
    coluna_idx = (mes - 1) % 3 + 1

    fig.add_trace(
        linha,
        row=linha_idx,
        col=coluna_idx
    )

    fig.add_trace(
        media_linha,
        row=linha_idx,
        col=coluna_idx
    )

fig.update_layout(
    title="Tendência das notificações por mês ao longo dos anos",
    height=900,
    width=1200
)

fig.show()

In [24]:
# Contagem dos resultados
dengue_counts = (
    classi_fin["DENGUE"]
    .value_counts()
    .reset_index()
)

dengue_counts.columns = ["Resultado", "Contagem"]

fig = px.bar(
    dengue_counts,
    x="Resultado",
    y="Contagem",
    text="Contagem",
    title="Número de casos de Dengue",
    labels={
        "Contagem": "Número de casos",
        "Resultado": "Resultado"
    },
    color="Resultado"
)

fig.update_traces(textposition="outside")

fig.show()

In [25]:
# Contagem dos resultados
raca_counts = (
    cs_raca["CS_RACA"]
    .value_counts()
    .reset_index()
)

raca_counts.columns = ["Raça", "Contagem"]

fig = px.bar(
    raca_counts,
    x="Raça",
    y="Contagem",
    text="Contagem",
    title="Número de casos de Dengue por Raça",
    labels={
        "Contagem": "Número de casos",
        "Raça": "Raça"
    },
    color="Raça"
)

fig.update_traces(textposition="outside")

fig.show()

In [26]:
# Contagem dos resultados
evolucao_counts = (
    df_evolucao["CON_EVOLUC"]
    .value_counts()
    .reset_index()
)

evolucao_counts.columns = ["Evolução", "Contagem"]

fig = px.bar(
    evolucao_counts,
    x="Evolução",
    y="Contagem",
    text="Contagem",
    title="Evolução dos casos de Dengue",
    labels={
        "Contagem": "Número de casos",
        "Evolução": "Evolução"
    },
    color="Evolução"
)

fig.update_traces(textposition="outside")

fig.show()

In [32]:
df_estado["Percentual"] = (
    df_estado["Notificações"]
    / df_estado["Notificações"].sum()
    * 100
)

df_estado = df_estado.sort_values(
    by="Notificações",
    ascending=False
)

fig = px.bar(
    df_estado,
    x="Estado",
    y="Notificações",
    color="Notificações",
    color_continuous_scale="Reds",
    text=df_estado["Percentual"].map(lambda x: f"{x:.1f}%"),
    title="Distribuição de Casos Positivos de Dengue por Estado",
)

fig.update_traces(textposition="outside")

fig.update_layout(
    xaxis_title="Estado (UF)",
    yaxis_title="Número de Casos",
    showlegend=False
)

fig.show()